# M20b — UCI Appliances Energy Prediction

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Apply the same four structural paths to the Appliances Energy time series.

**Provenance.** The original June 2026 notebook binary is unavailable. This notebook is a transparent reconstruction from the recovered manuscript settings and implementation lineage. It must therefore be interpreted as a fresh protocol replication, not as a recovery of the historical experiment.

**v18.3 execution repair.** The real-data studies use the manuscript-native absolute safe-region tolerance `epsilon_abs=0.002`, passed explicitly below. Fresh reconstruction outputs are written only to `results/reproduced/`; publication-facing historical values in `manuscript/tables/` are not overwritten.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import tcr_core as tcr

REPRO = ROOT / "results" / "reproduced"
REPRO.mkdir(parents=True, exist_ok=True)
PUB_TABLE = ROOT / "manuscript" / "tables" / "table_real_world.csv"

In [2]:
DATA = ROOT / "data" / "energydata_complete.csv"
URL = "https://archive.ics.uci.edu/static/public/374/appliances%2Benergy%2Bprediction.zip"
HORIZON = 6
N = 50
K = 13
RIDGE = 1e-4
SEED = 20260718
ABS_TOL = 0.002

pub = pd.read_csv(PUB_TABLE)
display(pub[pub["Dataset"] == "Appliances Energy"])
print("data file:", DATA)
print("present:", DATA.exists())
print("native v18.3 epsilon_abs:", ABS_TOL)

,Dataset,Rows/features,Target,Temperature support range,Temperature full-test best-safe gain,Temperature safe - shuffled gain,Temperature path indicators
0,Appliances Energy,19729/32,"log future appliances, h=6",45.62,0.00014,"-0.01199 [-0.05331, 0.01532]",0.510 / 0.522


data file: /mnt/data/tcr_repair/repo/data/energydata_complete.csv
present: True
native v18.3 epsilon_abs: 0.002


## Preprocessing

In [3]:
def prepare_appliances(path):
    df = pd.read_csv(path)
    dt = pd.to_datetime(df["date"])
    numeric = df.drop(columns=["date"]).apply(pd.to_numeric, errors="coerce")

    # All 28 contemporaneous numeric measurements are retained, including current Appliances.
    X = numeric.copy()
    X["hour_sin"] = np.sin(2*np.pi*dt.dt.hour/24)
    X["hour_cos"] = np.cos(2*np.pi*dt.dt.hour/24)
    X["dow_sin"] = np.sin(2*np.pi*dt.dt.dayofweek/7)
    X["dow_cos"] = np.cos(2*np.pi*dt.dt.dayofweek/7)
    y = np.log1p(numeric["Appliances"].shift(-HORIZON))

    X = X.iloc[:-HORIZON].reset_index(drop=True)
    y = y.iloc[:-HORIZON].reset_index(drop=True)
    assert len(X) == 19729 and X.shape[1] == 32, (len(X), X.shape)
    assert not X.isna().any().any() and not y.isna().any()
    return X.to_numpy(float), y.to_numpy(float)

## Fresh reconstruction execution

In [4]:
if not DATA.exists():
    raise FileNotFoundError(f"Place the official UCI file at {DATA}")

X, y = prepare_appliances(DATA)
print("prepared:", X.shape, y.shape)

ntr, nv, nt = 2500, 1000, 1000
Xtr, Xv, Xt = X[:ntr], X[ntr:ntr+nv], X[ntr+nv:ntr+nv+nt]
ytr, yv, yt = y[:ntr], y[ntr:ntr+nv], y[ntr+nv:ntr+nv+nt]
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-12
Xtr = (Xtr-mu)/sd
Xv = (Xv-mu)/sd
Xt = (Xt-mu)/sd

rows = []
for path in ["temperature", "gain", "leak", "sparsity"]:
    rows.append(tcr.evaluate_arrays(
        Xtr, ytr, Xv, yv, Xt, yt,
        "appliances", 0, path, SEED, N, K, RIDGE,
        abs_tol=ABS_TOL,
    ))
rep = pd.DataFrame(rows)
rep.to_csv(REPRO / "m20b_replication_summary.csv", index=False)
display(rep.round(6))

prepared: (19729, 32) (19729,)


,task,trial,path,default_idx,safe_low_idx,safe_high_idx,safe_width,safe_width_fraction,near_contained,exact_contained,matched_near_rate,matched_oracle_rate,safe_gain,full_gain,matched_best_gain,safe_minus_matched_gain,default_test_nrmse,safe_best_test_nrmse,full_best_test_nrmse,support_range,spectral_radius_range
0,appliances,0,temperature,10,9,11,3,0.230769,0,0,0.363636,0.181818,0.022306,0.112840,0.061545,-0.039239,1.015277,0.992971,0.902437,45.674764,0.106282
1,appliances,0,gain,12,11,12,2,0.153846,1,1,0.166667,0.083333,0.000000,0.000000,-0.032105,0.032105,0.953209,0.953209,0.953209,0.000000,1.538341
2,appliances,0,leak,12,8,12,5,0.384615,1,1,0.222222,0.111111,0.000000,0.000000,-0.113334,0.113334,0.987933,0.987933,0.987933,0.000000,0.000000
3,appliances,0,sparsity,7,7,7,1,0.076923,0,0,0.076923,0.076923,0.000000,0.090759,0.010968,-0.010968,0.979269,0.979269,0.888510,48.000000,0.209153


## Publication-reference comparison

In [5]:
fresh = rep.loc[rep.path == "temperature"].iloc[0]
published = float(pub.loc[pub["Dataset"] == "Appliances Energy", "Temperature full-test best-safe gain"].iloc[0])
comparison = pd.DataFrame([{
    "study": "Appliances Energy",
    "published_v18_3_gain": published,
    "fresh_reconstruction_gain": float(fresh.safe_gain),
    "fresh_safe_width": f"{int(fresh.safe_width)}/{K}",
    "exact_point_estimate_reproduced": bool(np.isclose(float(fresh.safe_gain), published, atol=1e-8)),
}])
display(comparison)
print("Fresh reconstruction is provenance-separated and does not overwrite the publication table.")

,study,published_v18_3_gain,fresh_reconstruction_gain,fresh_safe_width,exact_point_estimate_reproduced
0,Appliances Energy,0.00014,0.022306,3/13,False


Fresh reconstruction is provenance-separated and does not overwrite the publication table.


Official source: UCI Appliances Energy Prediction. The dataset itself is not redistributed by the repository; see `data/README.md`.